# Week 1 -Churn Prediction Pipeline
## Day2 - Preprocessing, SMOTE and Model Training

**Goal:** Clean data. encode features. balance classes. train 3 models. cross validate

**Input:** WA_Fn-UseC_-Telco-Customer-Churn.csv

**Output:** Trained XGBoost model saved as model.pkl

**Author:** Martin James |github.com/M20Jay

## Section 1 - Import Libraries
We import all tools needed for preprocessing, balancing and model training

In [1]:
# Import libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split,StratifiedKFold,cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report,confusion_matrix,roc_auc_score
from imblearn.over_sampling import SMOTE
import joblib
import warnings
warnings.filterwarnings("ignore")

## Section 2- Load and Prepare Data
We load dataset from Day 1 and apply initial cleaning before preprocessing

In [16]:
# Load and Prepare Data
df= pd.read_csv('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
# Fix TotalCharges — convert from object to numeric
df['TotalCharges']= pd.to_numeric(df['TotalCharges'], errors= 'coerce')

# Drop rows with missing TotalCharges (only 11 rows)
print("Missing TotalCharges:", df['TotalCharges'].isnull().sum())
df.dropna(subset=['TotalCharges'],inplace =True)

#Encode target column with  - Yes = 1 , No =0
df['Churn']=df['Churn'].map({'Yes':1, 'No':0})

print("\nShape after cleaning:", df.shape)
print("\nChurn value counts:")
print(df['Churn'].value_counts())

Missing TotalCharges: 11

Shape after cleaning: (7032, 21)

Churn value counts:
Churn
0    5163
1    1869
Name: count, dtype: int64


## Section 3 — Feature Engineering
We recreate the 3 features from Day 1 and drop columns not needed for modelling.

In [ ]:
#  Feature Engineering

# Recreate features from Day 1
df['tenure_group'] = pd.cut(df['tenure'], bins=[0, 12, 24, 72],labels=['New', 'Mid', 'Loyal'])

df['charges_per_month'] = df['TotalCharges'] / (df['tenure'] + 1)

df['is_high_value'] = (df['MonthlyCharges'] > 70).astype(int)

# Drop columns not needed for modelling
df.drop(columns=['customerID'], inplace =True)
print("Features after engineering:")
print(df.columns.to_list())
print("\nShape :", df.shape)


Features after engineering:
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn', 'tenure_group', 'charges_per_month', 'is_high_value']

Shape : (7032, 23)


## Section 4 -Encode Categorical Variables
We convert all text columns to numbers so the model can process them

In [30]:
# Ecode categorical variables
le =LabelEncoder()

# Columns to encode
df.select_dtypes(include='object').columns.tolist()

# Convert tenure_group from category to string for encoding
df['tenure_group'] = df['tenure_group'].astype(str)

# Columns to encode
cat_cols =['gender', 'Partner','Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies','Contract', 'PaperlessBilling','PaymentMethod','tenure_group']


for col in cat_cols:
    df[col] = le.fit_transform(df[col].astype(str))



print("Encoding complete ✅")
print(df.dtypes)


Encoding complete ✅
gender                 int64
SeniorCitizen          int64
Partner                int64
Dependents             int64
tenure                 int64
PhoneService           int64
MultipleLines          int64
InternetService        int64
OnlineSecurity         int64
OnlineBackup           int64
DeviceProtection       int64
TechSupport            int64
StreamingTV            int64
StreamingMovies        int64
Contract               int64
PaperlessBilling       int64
PaymentMethod          int64
MonthlyCharges       float64
TotalCharges         float64
Churn                  int64
tenure_group           int64
charges_per_month    float64
is_high_value          int64
dtype: object


## Section 5 -Train Test Split
We separate features from target and split into training and test sets

In [38]:
# Train Test Split

#Separate features and target
X= df.drop(columns=['Churn'])
y=df['Churn']
# Split 80% train 20% test — stratified to preserve churn ratio
X_train,X_test,y_train,y_test= train_test_split(X,y, test_size=0.2,random_state=42, stratify=y)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)
print("\n Churn ratio in training set:")
print(y_train.value_counts(normalize=True).round(3)*100)


Training set: (5625, 22)
Test set: (1407, 22)

 Churn ratio in training set:
Churn
0    73.4
1    26.6
Name: proportion, dtype: float64


## Section 6 - SMOTE Application
We apply SMOTE to balance training set. SMOTE creates synthetic minority class samples so the model learns churners as well as non-churners

In [41]:
# SMOTE application
print("Before SMOTE:", y_train.value_counts())

smote=SMOTE(random_state=42)
X_train_bal,y_train_bal =smote.fit_resample(X_train,y_train)

print("\nAfter SMOTE:")
print(pd.Series(y_train_bal.value_counts()))
print("\n New training size:", X_train_bal.shape)



Before SMOTE: Churn
0    4130
1    1495
Name: count, dtype: int64

After SMOTE:
Churn
0    4130
1    4130
Name: count, dtype: int64

 New training size: (8260, 22)


## Section 7 - Train 3 Models
We train Logistic Regression, Random Forest and XGBoost on the balanced dataset.
We use cross validation to get more reliable scores.

In [46]:
#  Train 3 models

#Define models
models ={
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state =42),
    'Random Forest': RandomForestClassifier(n_estimators =100, random_state =42),
    'XGBoost' : XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss')
    }

#cross validate each model
cv=StratifiedKFold(n_splits=5, shuffle = True, random_state=42)

print("=== Cross validation results ===")
for name, model in models.items():
    scores = cross_val_score(model,X_train_bal,y_train_bal, cv=cv, scoring= 'roc_auc')
    print(f"{name}: AUC ={scores.mean():.4} +/- {scores.std():.4}")


=== Cross validation results ===
Logistic Regression: AUC =0.8914 +/- 0.003367
Random Forest: AUC =0.9294 +/- 0.002322
XGBoost: AUC =0.9285 +/- 0.002883


## Section 8- Final Model Training and Evaluation



In [54]:
# Train XGBoost
xgb_model= XGBClassifier(n_estimators =100, random_state=42, eval_metric='logloss')
xgb_model.fit(X_train_bal,y_train_bal)

# Predict on Test
y_pred=xgb_model.predict(X_test)
y_pred_proba =xgb_model.predict_proba(X_test)[:,1]


# Evaluate
print("=== XGBoost test results ===")
print(f" AUC: {roc_auc_score(y_test, y_pred_proba):.4f}")
print("\n Classification report:")
print(classification_report(y_test, y_pred))
print("\nConfusion matix")
print(confusion_matrix(y_test, y_pred))


=== XGBoost test results ===
 AUC: 0.8091

 Classification report:
              precision    recall  f1-score   support

           0       0.84      0.81      0.83      1033
           1       0.53      0.58      0.56       374

    accuracy                           0.75      1407
   macro avg       0.69      0.70      0.69      1407
weighted avg       0.76      0.75      0.76      1407


Confusion matix
[[841 192]
 [156 218]]


## Section 9 — Save Model
We save the trained XGBoost model to disk so it can be loaded by the Flask API in Day 3

In [55]:
# Save Model
joblib.dump(xgb_model,'../src/churn_model.pkl')
print("Model saved to src/churn_model.pkl ✅")

Model saved to src/churn_model.pkl ✅
